In [0]:
%pip install datasets transformers sentence-transformers faiss-cpu torch --quiet
dbutils.library.restartPython()

In [0]:
%pip install -q \
  "datasets==2.19.0" \
  "huggingface_hub==0.23.0" \
  "fsspec==2024.2.0"

dbutils.library.restartPython()

In [0]:
from datasets import load_dataset
import os, base64
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Load HF token from secret scope
hf_token_b64 = w.secrets.get_secret("vaidya-lipi", "hf_token").value
hf_token = base64.b64decode(hf_token_b64).decode("utf-8")

# This dataset has real doctor-patient conversations with structured annotations
# It's your training/seeding data AND your eval ground truth
ds = load_dataset("ekacare/Eka-IndicMTEB", "queries", split="test", token=hf_token)
print(f"Loaded {len(ds)} medical terms from Eka-IndicMTEB corpus")
print(ds[0])

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()

# This is your main table — every consultation creates one row
schema = StructType([
    StructField("record_id",        StringType(),    False),  # UUID
    StructField("patient_id",       StringType(),    False),  # entered by doctor
    StructField("doctor_id",        StringType(),    False),  # from session
    StructField("hospital_id",      StringType(),    True),
    StructField("timestamp",        TimestampType(), False),
    StructField("language_detected",StringType(),    True),   # hi-IN, en-IN etc.
    StructField("raw_transcript",   StringType(),    True),   # raw ASR output
    StructField("symptoms",         ArrayType(StringType()), True),  # extracted
    StructField("diagnosis",        StringType(),    True),
    StructField("medications",      ArrayType(StringType()), True),
    StructField("snomed_codes",     ArrayType(StringType()), True),  # from Parrotlet-e
    StructField("structured_note",  StringType(),    True),   # full JSON string
    StructField("soap_subjective",  StringType(),    True),
    StructField("soap_objective",   StringType(),    True),
    StructField("soap_assessment",  StringType(),    True),
    StructField("soap_plan",        StringType(),    True),
    StructField("is_anonymized",    BooleanType(),   True),
])

spark.sql("CREATE DATABASE IF NOT EXISTS workspace.vaidya")

# Create table with Change Data Feed enabled (required for Vector Search later)
spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.vaidya.patient_records (
        record_id STRING NOT NULL,
        patient_id STRING NOT NULL,
        doctor_id STRING NOT NULL,
        hospital_id STRING,
        timestamp TIMESTAMP NOT NULL,
        language_detected STRING,
        raw_transcript STRING,
        symptoms ARRAY<STRING>,
        diagnosis STRING,
        medications ARRAY<STRING>,
        snomed_codes ARRAY<STRING>,
        structured_note STRING,
        soap_subjective STRING,
        soap_objective STRING,
        soap_assessment STRING,
        soap_plan STRING,
        is_anonymized BOOLEAN DEFAULT false
    )
    USING DELTA
    TBLPROPERTIES (
        delta.enableChangeDataFeed = true,
        delta.feature.allowColumnDefaults = 'supported'
    )
""")
print("patient_records table created")

In [0]:
import uuid, json
from datetime import datetime, timedelta
import random

# Use EkaCare dataset terms to create realistic synthetic records
# These are real Indian medical terms verified by doctors
sample_terms = list(ds)[:50]  # first 50 terms from the corpus

synthetic_rows = []
doctors = ["DR001", "DR002", "DR003"]
hospitals = ["AIIMS_MUMBAI", "KEM_HOSPITAL", "LILAVATI"]
languages = ["hi-IN", "en-IN", "mr-IN"]

for i, term in enumerate(sample_terms[:20]):
    symptoms = [term["term"]]  # the medical term (in any Indian language)
    snomed = [term["concept_id"]]
    lang = term.get("language", "en")
    lang_code = f"{lang}-IN" if len(lang) == 2 else lang

    structured = {
        "symptoms": symptoms,
        "diagnosis": "Under evaluation",
        "medications": ["To be prescribed"],
        "plan": "Follow-up in 1 week"
    }

    synthetic_rows.append({
        "record_id": str(uuid.uuid4()),
        "patient_id": f"PAT{1000+i}",
        "doctor_id": random.choice(doctors),
        "hospital_id": random.choice(hospitals),
        "timestamp": datetime.now() - timedelta(hours=random.randint(0, 48)),
        "language_detected": lang_code,
        "raw_transcript": f"Patient presents with {term['term']}",
        "symptoms": symptoms,
        "diagnosis": None,
        "medications": [],
        "snomed_codes": snomed,
        "structured_note": json.dumps(structured),
        "soap_subjective": f"Patient reports: {term['term']}",
        "soap_objective": "Vitals stable. Further investigation needed.",
        "soap_assessment": "Pending diagnosis",
        "soap_plan": "Order investigations. Follow up in 1 week.",
        "is_anonymized": False,
    })

df = spark.createDataFrame(synthetic_rows, schema=schema)
df.write.mode("append").saveAsTable("workspace.vaidya.patient_records")
print(f"Seeded {len(synthetic_rows)} synthetic records")
spark.sql("SELECT doctor_id, count(*) as patients FROM workspace.vaidya.patient_records GROUP BY doctor_id").show()

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.vaidya.health_alerts (
        alert_id STRING NOT NULL,
        generated_at TIMESTAMP NOT NULL,
        alert_type STRING,
        insight_text STRING,
        top_symptoms ARRAY<STRING>,
        affected_hospitals ARRAY<STRING>,
        severity STRING
    )
    USING DELTA
""")
print("health_alerts table created")